In [9]:
# ==================================================
# HGT Training for RC Element Prediction
# ==================================================

# ## Cell 1: Setup and Imports
import os
import sys
import yaml
import logging
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import torch_geometric
from torch.serialization import add_safe_globals
import warnings
warnings.filterwarnings('ignore')

print(f"📁 Current directory: {os.getcwd()}")

# Add project paths
sys.path.append("..")
sys.path.append("../src")

# Basic logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)-8s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger("NOTEBOOK")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.style.use('default')
sns.set_palette("husl")
sns.set_context("notebook", font_scale=1.2)

print("✅ Setup complete")

📁 Current directory: /Users/sarayetel/Desktop/Research/Graph/RC-Element-Prediction-GNN/notebooks
✅ Setup complete


In [10]:
# ## Cell 2: Import Project Modules

print("Importing project modules...")

from src.models.hgt import HGT
from src.training.trainer import HGTrainer, DeviceManager

print("✅ Modules imported successfully")

Importing project modules...
✅ Modules imported successfully


In [11]:
# ## Cell 3: Load Configuration

print("Loading configurations...")

# Load base config
with open("../configs/base.yaml", "r") as f:
    base_config = yaml.safe_load(f)

# Load HGT config (create if not exists)
hgt_config_path = "../configs/hgt.yaml"
if os.path.exists(hgt_config_path):
    with open(hgt_config_path, "r") as f:
        hgt_config = yaml.safe_load(f)
else:
    print("⚠️ hgt.yaml not found, creating default config...")
    hgt_config = {
        'model': {
            'type': 'hgt',
            'hidden_channels': 128,
            'num_layers': 3,
            'num_heads': 4,
            'dropout': 0.3,
            'node_types': ['beam', 'column'],
            'output_dim': 2,
            'use_structural_encoding': True,
        },
        'training': {
            'epochs': 100,
            'learning_rate': 0.001,
            'weight_decay': 0.0001,
            'patience': 20,
            'scheduler_factor': 0.5,
            'scheduler_patience': 10,
            'grad_clip': 1.0,
            'k_folds': 5,
            'width_weight': 1.0,
            'height_weight': 1.0,
            'beam_weight': 1.0,
            'column_weight': 1.0,
        },
        'paths': {
            'checkpoints': 'checkpoints/hgt',
        }
    }
    # Save default config
    os.makedirs("../configs", exist_ok=True)
    with open(hgt_config_path, "w") as f:
        yaml.dump(hgt_config, f, default_flow_style=False)
    print("✅ Created default hgt.yaml")

# Merge configs
config = {**base_config, **hgt_config}

print("\n📋 Configuration Summary:")
print(f"  Model: {config['model']['type']}")
print(f"  Hidden channels: {config['model']['hidden_channels']}")
print(f"  Layers: {config['model']['num_layers']}")
print(f"  Attention heads: {config['model']['num_heads']}")
print(f"  Dropout: {config['model']['dropout']}")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  K-Folds: {config['training']['k_folds']}")

Loading configurations...

📋 Configuration Summary:
  Model: hgt
  Hidden channels: 128
  Layers: 3
  Attention heads: 4
  Dropout: 0.3
  Epochs: 100
  Learning rate: 0.001
  K-Folds: 5


In [12]:
# ## Cell 4: Load Graphs

print("Loading pre-built graphs...")

# Register safe globals for PyG HeteroData
add_safe_globals([
    torch_geometric.data.storage.BaseStorage,
    torch_geometric.data.storage.NodeStorage,
    torch_geometric.data.storage.EdgeStorage,
    torch_geometric.data.HeteroData
])

# Load graphs
graph_path = "../data/graphs/train_graphs.pt"
if not os.path.exists(graph_path):
    raise FileNotFoundError(f"Graph file not found: {graph_path}")

loaded_data = torch.load(graph_path, weights_only=False)

# Handle different formats of saved graphs
if isinstance(loaded_data, list):
    # Check if it's a list of tuples (sample_name, graph)
    if len(loaded_data) > 0 and isinstance(loaded_data[0], tuple):
        print("Detected format: List of (sample_name, graph) tuples")
        sample_names = [item[0] for item in loaded_data]
        graphs = [item[1] for item in loaded_data]
        
        # Attach sample names to graphs if not present
        for i, (name, graph) in enumerate(zip(sample_names, graphs)):
            if not hasattr(graph, 'sample_name'):
                graph.sample_name = name
    else:
        print("Detected format: List of graphs")
        graphs = loaded_data
elif isinstance(loaded_data, dict):
    print("Detected format: Dictionary of graphs")
    graphs = list(loaded_data.values())
else:
    graphs = loaded_data

print(f"\n✅ Loaded {len(graphs)} graphs")

# Helper function to safely get graph attributes
def get_graph_info(graph):
    """Safely extract graph information."""
    info = {
        'sample_name': 'Unknown',
        'node_types': [],
        'edge_types': [],
    }
    
    # Get sample name
    if hasattr(graph, 'sample_name'):
        info['sample_name'] = graph.sample_name
    elif isinstance(graph, dict) and 'sample_name' in graph:
        info['sample_name'] = graph['sample_name']
    
    # Get node types
    if hasattr(graph, 'node_types'):
        info['node_types'] = list(graph.node_types)
    else:
        # Try to infer node types from attributes
        possible_types = ['beam', 'column']
        for nt in possible_types:
            if hasattr(graph, nt):
                node_data = graph[nt] if not isinstance(graph, dict) else graph.get(nt, {})
                if hasattr(node_data, 'x') or (isinstance(node_data, dict) and 'x' in node_data):
                    info['node_types'].append(nt)
    
    # Get edge types
    if hasattr(graph, 'edge_types'):
        info['edge_types'] = list(graph.edge_types)
    elif hasattr(graph, 'edge_index_dict'):
        info['edge_types'] = list(graph.edge_index_dict.keys())
    
    return info

# Analyze dataset
print("\n📊 Dataset Overview:")
print(f"  Total samples: {len(graphs)}")

# Analyze first few graphs
for i, graph in enumerate(graphs[:3]):
    info = get_graph_info(graph)
    print(f"\n  Graph {i+1}: {info['sample_name']}")
    
    # Count nodes per type
    for node_type in info['node_types']:
        try:
            node_data = graph[node_type] if not isinstance(graph, dict) else graph[node_type]
            if hasattr(node_data, 'x'):
                x = node_data.x
            elif isinstance(node_data, dict) and 'x' in node_data:
                x = node_data['x']
            else:
                x = None
            
            if x is not None:
                num_nodes = x.shape[0]
                num_features = x.shape[1] if len(x.shape) > 1 else 1
                
                # Check for labels
                if hasattr(node_data, 'y'):
                    has_labels = node_data.y is not None
                elif isinstance(node_data, dict):
                    has_labels = 'y' in node_data and node_data['y'] is not None
                else:
                    has_labels = False
                
                print(f"    {node_type}: {num_nodes} nodes, {num_features} features, labels: {has_labels}")
        except Exception as e:
            print(f"    {node_type}: Error accessing - {e}")
    
    # Count edges per type
    for edge_type in info['edge_types']:
        try:
            if hasattr(graph, 'edge_index_dict'):
                edge_index = graph.edge_index_dict.get(edge_type)
            elif isinstance(graph, dict) and 'edge_index_dict' in graph:
                edge_index = graph['edge_index_dict'].get(edge_type)
            else:
                edge_index = None
            
            if edge_index is not None:
                num_edges = edge_index.shape[1] if hasattr(edge_index, 'shape') else len(edge_index[0])
                if num_edges > 0:
                    print(f"    {edge_type}: {num_edges} edges")
        except Exception as e:
            print(f"    {edge_type}: Error accessing - {e}")

# Overall statistics (robust version)
total_beams = 0
total_columns = 0
total_edges = 0

for graph in graphs:
    info = get_graph_info(graph)
    
    # Count beam nodes
    if 'beam' in info['node_types']:
        try:
            node_data = graph['beam'] if not isinstance(graph, dict) else graph['beam']
            x = node_data.x if hasattr(node_data, 'x') else node_data.get('x')
            if x is not None:
                total_beams += x.shape[0]
        except:
            pass
    
    # Count column nodes
    if 'column' in info['node_types']:
        try:
            node_data = graph['column'] if not isinstance(graph, dict) else graph['column']
            x = node_data.x if hasattr(node_data, 'x') else node_data.get('x')
            if x is not None:
                total_columns += x.shape[0]
        except:
            pass
    
    # Count edges
    for edge_type in info['edge_types']:
        try:
            if hasattr(graph, 'edge_index_dict'):
                edge_index = graph.edge_index_dict.get(edge_type)
            elif isinstance(graph, dict) and 'edge_index_dict' in graph:
                edge_index = graph['edge_index_dict'].get(edge_type)
            else:
                edge_index = None
            
            if edge_index is not None:
                total_edges += edge_index.shape[1] if hasattr(edge_index, 'shape') else len(edge_index[0])
        except:
            pass

print(f"\n📈 Total Statistics:")
print(f"  Total beams: {total_beams}")
print(f"  Total columns: {total_columns}")
print(f"  Total nodes: {total_beams + total_columns}")
print(f"  Total edges: {total_edges}")
print(f"  Avg nodes/graph: {(total_beams + total_columns) / len(graphs):.1f}" if len(graphs) > 0 else "  No graphs loaded")
print(f"  Avg edges/graph: {total_edges / len(graphs):.1f}" if len(graphs) > 0 else "  No graphs loaded")

# Validate graphs for training
print("\n🔍 Validating graphs for training...")
valid_graphs = []
invalid_count = 0

for i, graph in enumerate(graphs):
    info = get_graph_info(graph)
    is_valid = True
    issues = []
    
    # Check required node types
    if 'beam' not in info['node_types']:
        issues.append("Missing 'beam' node type")
        is_valid = False
    if 'column' not in info['node_types']:
        issues.append("Missing 'column' node type")
        is_valid = False
    
    # Check features
    for node_type in info['node_types']:
        try:
            node_data = graph[node_type] if not isinstance(graph, dict) else graph[node_type]
            x = node_data.x if hasattr(node_data, 'x') else node_data.get('x')
            if x is None or x.shape[0] == 0:
                issues.append(f"No features for {node_type}")
                is_valid = False
        except:
            issues.append(f"Cannot access {node_type} features")
            is_valid = False
    
    # Check edges
    if len(info['edge_types']) == 0:
        issues.append("No edge types found")
        is_valid = False
    
    if is_valid:
        valid_graphs.append(graph)
    else:
        invalid_count += 1
        if i < 3:  # Only print first few invalid graphs
            print(f"  ⚠️ Graph {i+1} ({info['sample_name']}): {', '.join(issues)}")

# Update graphs list
if invalid_count > 0:
    print(f"\n  ⚠️ Found {invalid_count} invalid graphs (removing from dataset)")
    graphs = valid_graphs
    print(f"  ✅ Using {len(graphs)} valid graphs for training")
else:
    print(f"  ✅ All {len(graphs)} graphs are valid for training")

Loading pre-built graphs...
Detected format: List of (sample_name, graph) tuples

✅ Loaded 254 graphs

📊 Dataset Overview:
  Total samples: 254

  Graph 1: sample_1
    beam: 371 nodes, 49 features, labels: True
    column: 103 nodes, 49 features, labels: True
    ('beam', 'to', 'beam'): 1296 edges
    ('column', 'to', 'column'): 174 edges
    ('beam', 'to', 'column'): 1334 edges
    ('column', 'to', 'beam'): 1334 edges

  Graph 2: sample_10
    beam: 176 nodes, 49 features, labels: True
    column: 96 nodes, 49 features, labels: True
    ('beam', 'to', 'beam'): 692 edges
    ('column', 'to', 'column'): 162 edges
    ('beam', 'to', 'column'): 1092 edges
    ('column', 'to', 'beam'): 1092 edges

  Graph 3: sample_100
    beam: 205 nodes, 49 features, labels: True
    column: 109 nodes, 49 features, labels: True
    ('beam', 'to', 'beam'): 652 edges
    ('column', 'to', 'column'): 188 edges
    ('beam', 'to', 'column'): 1188 edges
    ('column', 'to', 'beam'): 1188 edges

📈 Total Statist

In [13]:
# ## Cell 5: Check Device Availability

print("Checking device availability...\n")

# Check CUDA
if torch.cuda.is_available():
    print(f"✅ CUDA available: {torch.cuda.get_device_name()}")
    print(f"   CUDA version: {torch.version.cuda}")
else:
    print("❌ CUDA not available")

# Check MPS (Apple Silicon)
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print(f"✅ MPS available (Apple Silicon)")
else:
    print("❌ MPS not available")

# Get device
device = DeviceManager.get_device()
print(f"\n🎯 Using device: {device}")

# Quick test
try:
    test_tensor = torch.tensor([1.0, 2.0, 3.0]).to(device)
    print(f"✅ Device test passed: {test_tensor.device}")
except Exception as e:
    print(f"❌ Device test failed: {e}")

15:26:07 | TRAINER  | INFO     | Using Apple MPS (Metal Performance Shaders)


Checking device availability...

❌ CUDA not available
✅ MPS available (Apple Silicon)

🎯 Using device: mps
✅ Device test passed: mps:0


In [14]:
# ## Cell 6: Create HGT Model

print("Creating HGT model...\n")

model_config = config['model']

# Create model
model = HGT(
    hidden_channels=model_config['hidden_channels'],
    num_layers=model_config['num_layers'],
    num_heads=model_config['num_heads'],
    dropout=model_config['dropout'],
    node_types=model_config.get('node_types', ['beam', 'column']),
    output_dim=model_config.get('output_dim', 2),
    use_structural_encoding=model_config.get('use_structural_encoding', True),
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Model Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")

# Test forward pass with first graph
print(f"\n🔍 Testing forward pass...")
try:
    test_graph = graphs[0]
    model.eval()
    with torch.no_grad():
        predictions = model(test_graph)
    
    print("✅ Forward pass successful!")
    print(f"\n  Predictions:")
    for node_type, pred in predictions.items():
        print(f"    {node_type}: shape {pred.shape} (min={pred.min().item():.2f}, max={pred.max().item():.2f})")
    
    # Check against labels
    if hasattr(test_graph['beam'], 'y'):
        print(f"\n  Ground truth:")
        for node_type in ['beam', 'column']:
            if hasattr(test_graph[node_type], 'y'):
                y = test_graph[node_type].y
                print(f"    {node_type}: shape {y.shape} (min={y.min().item():.2f}, max={y.max().item():.2f})")
except Exception as e:
    print(f"❌ Forward pass failed: {e}")
    import traceback
    traceback.print_exc()

15:26:07 | HGT      | INFO     | HGT initialized: 3 layers, 4 heads, 128 hidden
15:26:07 | HGT      | INFO     | Initializing HGT with metadata: (['beam', 'column'], [('beam', 'to', 'beam'), ('column', 'to', 'column'), ('beam', 'to', 'column'), ('column', 'to', 'beam')])


Creating HGT model...

📊 Model Statistics:
  Total parameters: 0
  Trainable parameters: 0
  Model size: 0.00 MB (float32)

🔍 Testing forward pass...
❌ Forward pass failed: name 'hidden_channels' is not defined


Traceback (most recent call last):
  File "/var/folders/nv/nf693pxs72lbcg7sm8bhb3dw0000gn/T/ipykernel_12168/267039621.py", line 33, in <module>
    predictions = model(test_graph)
                  ^^^^^^^^^^^^^^^^^
  File "/Users/sarayetel/anaconda3/envs/Project/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sarayetel/anaconda3/envs/Project/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1784, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sarayetel/Desktop/Research/Graph/RC-Element-Prediction-GNN/notebooks/../src/models/hgt.py", line 182, in forward
    self._initialize_from_graph(graph)
  File "/Users/sarayetel/Desktop/Research/Graph/RC-Element-Prediction-GNN/notebooks/../src/models/hgt.py", line 103, in _initialize_from_graph
    Linear(in_channels, hidden_cha

In [15]:
# ## Cell 7: Split Data into Train/Val/Test

print("Splitting data into train/validation/test sets...\n")

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Shuffle indices
n_graphs = len(graphs)
indices = np.random.permutation(n_graphs)

# Split ratios
train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

train_end = int(n_graphs * train_ratio)
val_end = int(n_graphs * (train_ratio + val_ratio))

train_indices = indices[:train_end]
val_indices = indices[train_end:val_end]
test_indices = indices[val_end:]

train_graphs = [graphs[i] for i in train_indices]
val_graphs = [graphs[i] for i in val_indices]
test_graphs = [graphs[i] for i in test_indices]

print(f"📊 Data Split:")
print(f"  Train: {len(train_graphs)} graphs ({len(train_graphs)/n_graphs*100:.1f}%)")
print(f"  Validation: {len(val_graphs)} graphs ({len(val_graphs)/n_graphs*100:.1f}%)")
print(f"  Test: {len(test_graphs)} graphs ({len(test_graphs)/n_graphs*100:.1f}%)")

# Show label statistics for each split
def show_label_stats(graphs_list, name):
    beam_widths = []
    beam_heights = []
    col_widths = []
    col_heights = []
    
    for g in graphs_list:
        if hasattr(g['beam'], 'y'):
            y = g['beam'].y
            beam_widths.extend(y[:, 0].tolist())
            beam_heights.extend(y[:, 1].tolist())
        if hasattr(g['column'], 'y'):
            y = g['column'].y
            col_widths.extend(y[:, 0].tolist())
            col_heights.extend(y[:, 1].tolist())
    
    stats = pd.DataFrame({
        'Beam Width': [np.mean(beam_widths), np.std(beam_widths), np.min(beam_widths), np.max(beam_widths)],
        'Beam Height': [np.mean(beam_heights), np.std(beam_heights), np.min(beam_heights), np.max(beam_heights)],
        'Column Width': [np.mean(col_widths), np.std(col_widths), np.min(col_widths), np.max(col_widths)],
        'Column Height': [np.mean(col_heights), np.std(col_heights), np.min(col_heights), np.max(col_heights)],
    }, index=['Mean', 'Std', 'Min', 'Max'])
    
    print(f"\n{name} Label Statistics:")
    print(stats.to_string(float_format=lambda x: f'{x:.4f}'))

show_label_stats(train_graphs, "Training")
show_label_stats(val_graphs, "Validation")
show_label_stats(test_graphs, "Test")

Splitting data into train/validation/test sets...

📊 Data Split:
  Train: 177 graphs (69.7%)
  Validation: 38 graphs (15.0%)
  Test: 39 graphs (15.4%)

Training Label Statistics:
      Beam Width  Beam Height  Column Width  Column Height
Mean     45.9534      39.8133       53.5681        52.2847
Std      16.6204      14.9677        9.0629         7.3548
Min       0.0000       0.0000        0.0000         0.0000
Max      90.0000      75.0000       80.0000        75.0000

Validation Label Statistics:
      Beam Width  Beam Height  Column Width  Column Height
Mean     44.0121      38.3928       50.4168        50.1001
Std      15.2479      13.6789        7.3969         7.0607
Min       0.0000       0.0000        0.0000         0.0000
Max      80.0000      70.0000       75.0000        75.0000

Test Label Statistics:
      Beam Width  Beam Height  Column Width  Column Height
Mean     46.0901      40.1678       55.6958        52.4474
Std      18.0202      15.9236       17.1574         9.5848


In [17]:
# ## Cell 8: Initialize Trainer and Start Training

print("Initializing trainer...\n")

# Recreate model (fresh weights)
model = HGT(
    hidden_channels=config['model']['hidden_channels'],
    num_layers=config['model']['num_layers'],
    num_heads=config['model']['num_heads'],
    dropout=config['model']['dropout'],
    node_types=config['model'].get('node_types', ['beam', 'column']),
    output_dim=config['model'].get('output_dim', 2),
    use_structural_encoding=config['model'].get('use_structural_encoding', True),
)

# Create trainer
trainer = HGTrainer(
    model=model,
    config=config,
    device=device,
)

print("✅ Trainer initialized")
print(f"  Device: {trainer.device}")
print(f"  Epochs: {trainer.epochs}")
print(f"  Learning rate: {trainer.learning_rate}")
print(f"  Early stopping patience: {trainer.patience}")

# ## Test forward pass before training
print(f"\n{'='*60}")
print("Testing model forward pass before training...")
print(f"{'='*60}")

test_graph = all_train_val_graphs[0].clone() if hasattr(all_train_val_graphs[0], 'clone') else all_train_val_graphs[0]
test_graph = test_graph.to(device)

model.train()
try:
    with torch.no_grad():
        predictions = model(test_graph)
    print("✅ Forward pass successful!")
    for node_type, pred in predictions.items():
        print(f"  {node_type}: {pred.shape} on {pred.device}")
except Exception as e:
    print(f"❌ Forward pass failed: {e}")
    print("\nTrying with CPU instead...")
    # Fall back to CPU if MPS fails
    model = model.cpu()
    test_graph = test_graph.cpu()
    device = torch.device("cpu")
    trainer.device = device
    trainer.model = model
    
    with torch.no_grad():
        predictions = model(test_graph)
    print("✅ Forward pass successful on CPU!")
    for node_type, pred in predictions.items():
        print(f"  {node_type}: {pred.shape} on {pred.device}")

# ## Start Training with K-Fold Cross Validation
print(f"\n{'='*60}")
print(f"Starting K-Fold Cross Validation Training")
print(f"{'='*60}")

# Determine number of folds based on data size
n_graphs_total = len(train_graphs) + len(val_graphs)
n_folds = min(config['training'].get('k_folds', 5), n_graphs_total // 2)
n_folds = max(2, n_folds)  # At least 2 folds

print(f"Using {n_folds} folds (based on {n_graphs_total} total graphs)")

# Combine train and val for k-fold (we'll split internally)
all_train_val_graphs = train_graphs + val_graphs

# Train with k-fold
try:
    cv_results = trainer.train_with_kfold(
        graphs=all_train_val_graphs,
        n_folds=n_folds,
        shuffle=True,
        random_state=42,
    )
    print(f"\n✅ Training complete!")
    
except Exception as e:
    print(f"\n❌ Training failed with error: {e}")
    import traceback
    traceback.print_exc()
    
    # Try with simpler training approach
    print(f"\n{'='*60}")
    print("Attempting simple train/val split training instead...")
    print(f"{'='*60}")
    
    # Reset model
    model = HGT(
        hidden_channels=config['model']['hidden_channels'],
        num_layers=config['model']['num_layers'],
        num_heads=config['model']['num_heads'],
        dropout=config['model']['dropout'],
        node_types=config['model'].get('node_types', ['beam', 'column']),
        output_dim=config['model'].get('output_dim', 2),
        use_structural_encoding=config['model'].get('use_structural_encoding', True),
    )
    
    trainer = HGTrainer(model=model, config=config, device=device)
    
    cv_results = trainer.train_simple(
        train_graphs=train_graphs,
        val_graphs=val_graphs,
    )
    print(f"\n✅ Simple training complete!")

15:26:18 | HGT      | INFO     | HGT initialized: 3 layers, 4 heads, 128 hidden
15:26:18 | TRAINER  | INFO     | Trainer initialized on device: mps
15:26:18 | TRAINER  | INFO     | Training config: epochs=100, lr=0.001, folds=5


Initializing trainer...

✅ Trainer initialized
  Device: mps
  Epochs: 100
  Learning rate: 0.001
  Early stopping patience: 20

Testing model forward pass before training...


NameError: name 'all_train_val_graphs' is not defined

In [ ]:
# ## Cell 9: Visualize Training History

print("Visualizing training history...\n")

# Extract metrics from the best fold
best_fold_idx = np.argmin([fold['best_val_loss'] for fold in cv_results['fold_results']])
best_fold = cv_results['fold_results'][best_fold_idx]
fold_metrics = best_fold['metrics']

print(f"Best fold: {best_fold_idx + 1} (val_loss: {best_fold['best_val_loss']:.4f})")

# Plot training curves
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Training and Validation Loss
ax = axes[0, 0]
epochs = range(1, len(fold_metrics['train_loss']) + 1)
ax.plot(epochs, fold_metrics['train_loss'], label='Train Loss', linewidth=2)
ax.plot(epochs, fold_metrics['val_loss'], label='Val Loss', linewidth=2)
ax.axvline(x=best_fold['best_epoch'], color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_fold["best_epoch"]})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Learning Rate
ax = axes[0, 1]
ax.plot(epochs, fold_metrics['lr'], color='purple', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# 3. Beam MAE
ax = axes[0, 2]
if 'val_beam_mae' in fold_metrics:
    ax.plot(epochs, fold_metrics['val_beam_mae'], label='Beam MAE', color='orange', linewidth=2)
if 'val_column_mae' in fold_metrics:
    ax.plot(epochs, fold_metrics['val_column_mae'], label='Column MAE', color='green', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE')
ax.set_title('Validation MAE by Node Type')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Width MAE
ax = axes[1, 0]
if 'val_beam_width_mae' in fold_metrics:
    ax.plot(epochs, fold_metrics['val_beam_width_mae'], label='Beam Width', color='red', linewidth=2)
if 'val_column_width_mae' in fold_metrics:
    ax.plot(epochs, fold_metrics['val_column_width_mae'], label='Column Width', color='blue', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE')
ax.set_title('Validation Width MAE')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Height MAE
ax = axes[1, 1]
if 'val_beam_height_mae' in fold_metrics:
    ax.plot(epochs, fold_metrics['val_beam_height_mae'], label='Beam Height', color='darkred', linewidth=2)
if 'val_column_height_mae' in fold_metrics:
    ax.plot(epochs, fold_metrics['val_column_height_mae'], label='Column Height', color='darkblue', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE')
ax.set_title('Validation Height MAE')
ax.legend()
ax.grid(True, alpha=0.3)

# 6. Cross-validation summary
ax = axes[1, 2]
fold_nums = [f['fold'] for f in cv_results['fold_results']]
fold_losses = [f['best_val_loss'] for f in cv_results['fold_results']]
ax.bar(fold_nums, fold_losses, color='skyblue', edgecolor='navy')
ax.axhline(y=cv_results['mean_best_val_loss'], color='red', linestyle='--', 
           label=f"Mean: {cv_results['mean_best_val_loss']:.4f}")
ax.set_xlabel('Fold')
ax.set_ylabel('Best Val Loss')
ax.set_title('Cross-Validation Results')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'HGT Training History - {n_folds}-Fold Cross Validation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print fold summary
print(f"\n📊 Cross-Validation Summary:")
print(f"  Mean Val Loss: {cv_results['mean_best_val_loss']:.4f} ± {cv_results['std_best_val_loss']:.4f}")
print(f"  Mean Best Epoch: {cv_results['mean_best_epoch']:.1f}")

In [ ]:
# ## Cell 10: Load Best Model and Evaluate on Test Set

print("Loading best model and evaluating on test set...\n")

# Load the best model from k-fold
best_model = trainer.load_best_model(fold=best_fold_idx)

# Evaluate on test set
test_results = trainer.evaluate(test_graphs)

# Extract metrics
test_metrics = test_results['metrics']

# Create results table
results_data = []
for metric_name, values in test_metrics.items():
    results_data.append({
        'Metric': metric_name,
        'Mean': values['mean'],
        'Std': values['std'],
    })

results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values('Metric')

print("\n📊 Test Set Results:")
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
# ## Cell 11: Detailed Prediction Analysis

print("Analyzing predictions in detail...\n")

# Collect all predictions and ground truth
all_beam_preds = []
all_beam_true = []
all_column_preds = []
all_column_true = []

for graph, pred_dict in zip(test_graphs, test_results['predictions']):
    for node_type, pred in pred_dict.items():
        if hasattr(graph[node_type], 'y'):
            true = graph[node_type].y.cpu().numpy()
            
            if node_type == 'beam':
                all_beam_preds.append(pred)
                all_beam_true.append(true)
            elif node_type == 'column':
                all_column_preds.append(pred)
                all_column_true.append(true)

# Concatenate
if all_beam_preds:
    beam_preds = np.concatenate(all_beam_preds, axis=0)
    beam_true = np.concatenate(all_beam_true, axis=0)
else:
    beam_preds = np.array([])
    beam_true = np.array([])

if all_column_preds:
    column_preds = np.concatenate(all_column_preds, axis=0)
    column_true = np.concatenate(all_column_true, axis=0)
else:
    column_preds = np.array([])
    column_true = np.array([])

# Create scatter plots
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

plot_data = [
    (0, 0, beam_true[:, 0], beam_preds[:, 0], 'Beam Width', 'orange'),
    (0, 1, beam_true[:, 1], beam_preds[:, 1], 'Beam Height', 'darkorange'),
    (1, 0, column_true[:, 0], column_preds[:, 0], 'Column Width', 'green'),
    (1, 1, column_true[:, 1], column_preds[:, 1], 'Column Height', 'darkgreen'),
]

for row, col, true_vals, pred_vals, title, color in plot_data:
    if len(true_vals) == 0:
        continue
    
    ax = axes[row, col]
    
    # Scatter plot
    ax.scatter(true_vals, pred_vals, alpha=0.5, color=color, edgecolors='black', linewidth=0.5)
    
    # Perfect prediction line
    min_val = min(true_vals.min(), pred_vals.min())
    max_val = max(true_vals.max(), pred_vals.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    # R² score
    ss_res = np.sum((true_vals - pred_vals) ** 2)
    ss_tot = np.sum((true_vals - np.mean(true_vals)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    
    # MAE
    mae = np.mean(np.abs(true_vals - pred_vals))
    
    ax.set_xlabel('True Values')
    ax.set_ylabel('Predicted Values')
    ax.set_title(f'{title}\nMAE: {mae:.4f}, R²: {r2:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('HGT Predictions vs Ground Truth on Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Error distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

error_data = [
    (0, 0, beam_true[:, 0] - beam_preds[:, 0], 'Beam Width Error', 'orange'),
    (0, 1, beam_true[:, 1] - beam_preds[:, 1], 'Beam Height Error', 'darkorange'),
    (1, 0, column_true[:, 0] - column_preds[:, 0], 'Column Width Error', 'green'),
    (1, 1, column_true[:, 1] - column_preds[:, 1], 'Column Height Error', 'darkgreen'),
]

for row, col, errors, title, color in error_data:
    if len(errors) == 0:
        continue
    
    ax = axes[row, col]
    ax.hist(errors, bins=30, color=color, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
    ax.axvline(x=np.mean(errors), color='blue', linestyle='-', linewidth=2, label=f'Mean: {np.mean(errors):.4f}')
    ax.set_xlabel('Error (True - Predicted)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{title}\nStd: {np.std(errors):.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Prediction Error Distribution on Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ## Cell 12: Per-Sample Analysis

print("Per-sample performance analysis...\n")

# Calculate metrics per sample
sample_metrics = []

for i, (graph, pred_dict) in enumerate(zip(test_graphs, test_results['predictions'])):
    sample_name = graph.sample_name if hasattr(graph, 'sample_name') else f'Sample_{i}'
    
    metrics_dict = {'sample': sample_name}
    
    for node_type in ['beam', 'column']:
        if node_type in pred_dict and hasattr(graph[node_type], 'y'):
            pred = pred_dict[node_type]
            true = graph[node_type].y.cpu().numpy()
            
            if len(true) > 0:
                mae = np.mean(np.abs(pred - true))
                metrics_dict[f'{node_type}_mae'] = mae
                
                # Per dimension
                for dim, dim_name in enumerate(['width', 'height']):
                    if true.shape[1] > dim:
                        dim_mae = np.mean(np.abs(pred[:, dim] - true[:, dim]))
                        metrics_dict[f'{node_type}_{dim_name}_mae'] = dim_mae
    
    sample_metrics.append(metrics_dict)

# Create DataFrame
sample_df = pd.DataFrame(sample_metrics)

print("Sample Performance Summary:")
print(sample_df.describe().to_string(float_format=lambda x: f'{x:.4f}'))

# Plot per-sample performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Beam MAE per sample
if 'beam_mae' in sample_df.columns:
    ax = axes[0]
    samples = range(len(sample_df))
    ax.bar(samples, sample_df['beam_mae'], color='orange', alpha=0.7, label='Beam')
    ax.axhline(y=sample_df['beam_mae'].mean(), color='red', linestyle='--', 
               label=f"Mean: {sample_df['beam_mae'].mean():.4f}")
    ax.set_xlabel('Sample')
    ax.set_ylabel('MAE')
    ax.set_title('Beam MAE per Sample')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Column MAE per sample
if 'column_mae' in sample_df.columns:
    ax = axes[1]
    ax.bar(samples, sample_df['column_mae'], color='green', alpha=0.7, label='Column')
    ax.axhline(y=sample_df['column_mae'].mean(), color='red', linestyle='--',
               label=f"Mean: {sample_df['column_mae'].mean():.4f}")
    ax.set_xlabel('Sample')
    ax.set_ylabel('MAE')
    ax.set_title('Column MAE per Sample')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Per-Sample Prediction Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ## Cell 13: Save Results

print("Saving results...\n")

# Create results directory
results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save test metrics
test_metrics_serializable = {}
for key, value in test_metrics.items():
    test_metrics_serializable[key] = {
        'mean': float(value['mean']),
        'std': float(value['std']),
    }

results = {
    'timestamp': timestamp,
    'config': {
        'model': config['model'],
        'training': config['training'],
    },
    'cv_results': {
        'n_folds': cv_results['n_folds'],
        'mean_val_loss': float(cv_results['mean_best_val_loss']),
        'std_val_loss': float(cv_results['std_best_val_loss']),
    },
    'test_metrics': test_metrics_serializable,
}

# Save to JSON
results_path = results_dir / f"hgt_results_{timestamp}.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

# Save test predictions
predictions_path = results_dir / f"hgt_predictions_{timestamp}.pt"
torch.save(test_results['predictions'], predictions_path)

print(f"✅ Results saved:")
print(f"  Metrics: {results_path}")
print(f"  Predictions: {predictions_path}")

# Also save to latest files (overwrite)
latest_metrics_path = results_dir / "latest_metrics.json"
with open(latest_metrics_path, 'w') as f:
    json.dump(results, f, indent=2)

latest_pred_path = results_dir / "latest_predictions.pt"
torch.save(test_results['predictions'], latest_pred_path)

print(f"  Latest metrics: {latest_metrics_path}")
print(f"  Latest predictions: {latest_pred_path}")

In [ ]:
# ## Cell 14: Summary and Conclusions

print("=" * 60)
print("HGT Training Summary")
print("=" * 60)

print(f"\n📋 Model Configuration:")
print(f"  Architecture: HGT with {config['model']['num_layers']} layers")
print(f"  Hidden channels: {config['model']['hidden_channels']}")
print(f"  Attention heads: {config['model']['num_heads']}")
print(f"  Dropout: {config['model']['dropout']}")

print(f"\n📊 Dataset:")
print(f"  Total graphs: {len(graphs)}")
print(f"  Training: {len(train_graphs)}")
print(f"  Validation: {len(val_graphs)}")
print(f"  Test: {len(test_graphs)}")

print(f"\n🎯 Cross-Validation Results:")
print(f"  Folds: {cv_results['n_folds']}")
print(f"  Mean Val Loss: {cv_results['mean_best_val_loss']:.4f} ± {cv_results['std_best_val_loss']:.4f}")

print(f"\n📈 Test Results:")
for metric_name, values in test_metrics.items():
    print(f"  {metric_name}: {values['mean']:.4f} ± {values['std']:.4f}")

print(f"\n💾 Results saved to: ../results/")
print(f"\n✅ Training pipeline complete!")